# Step 2. RAG Agent

내부 문서에서 실제로 근거를 회수할 수 있는지 검증한다.
초기 실험은 외부 임베딩 다운로드 없이 `TF-IDF + chunk retrieval`로 진행한다.


In [ ]:
from pathlib import Path
import sys
from dotenv import load_dotenv

CURRENT = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [CURRENT, *CURRENT.parents]:
    if (candidate / "src" / "semiconductor_agent").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError("project root with src/semiconductor_agent not found")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

for env_candidate in [PROJECT_ROOT / ".env", PROJECT_ROOT.parent / ".env"]:
    if env_candidate.exists():
        load_dotenv(env_candidate, override=False)
import pandas as pd

from semiconductor_agent.references import build_reference_inventory
from semiconductor_agent.rag import load_local_retrieved_docs, LocalTfidfRetriever


## 2-1. Local Reference Inventory

현재 확보된 PDF/웹페이지 PDF 자료를 인벤토리화한다.


In [ ]:
inventory = build_reference_inventory()
pd.DataFrame(inventory).head(20)


## 2-2. Load Chunked Documents

PDF를 페이지별로 읽고 chunk 단위로 분해한다.


In [ ]:
docs = load_local_retrieved_docs(max_pdfs=8, max_pages_per_pdf=6, chunk_size=900, chunk_overlap=150)
print("chunk_count:", len(docs))
pd.DataFrame(docs).head(10)


## 2-3. Build Local Retriever

로컬 lexical retriever를 구성해 초기 검색 품질을 확인한다.


In [ ]:
retriever = LocalTfidfRetriever.from_docs(docs)
print("retriever_ready:", len(retriever.docs))


## 2-4. Query Test

HBM4, PIM, CXL 관련 쿼리별로 어떤 chunk가 회수되는지 본다.


In [ ]:
test_queries = [
    "HBM4 hybrid bonding roadmap",
    "AiMX PIM LLM inference",
    "CXL DDR5 customer validation",
]

for query in test_queries:
    print("=" * 100)
    print("QUERY:", query)
    for item in retriever.search(query, top_k=5):
        print(f"- score={item['score']} | page={item['page']} | {item['source_title']}")
        print(f"  preview={item['content_preview'][:160]}...")


## 2-5. Next Work

다음 단계에서는 embedding 기반 검색, QA 평가셋, source filtering을 붙인다.


In [ ]:
next_tasks = [
    "TF-IDF baseline과 embedding retrieval 비교",
    "문서 타입별 source filtering",
    "QA set 기반 Hit Rate@K / MRR 측정",
    "rag_evidence 구조로 변환하는 formatter 추가",
]
next_tasks
